# H-Neurons small Qwen experiment

This notebook is configured for Colab/Kaggle with a single T4 GPU. It uses a Hugging Face model ID directly, so you do not need to download the model manually first.

In [1]:
import os
import pathlib
import subprocess
import sys

In [2]:
# Now run your git clone command again
!git clone -b dev --single-branch https://github.com/CallmeAndree/H-Neuron-Implementation.git

Cloning into 'H-Neuron-Implementation'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 148 (delta 51), reused 106 (delta 43), pack-reused 31 (from 1)
Receiving objects: 100% (148/148), 54.05 MiB | 28.71 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [3]:
os.chdir('H-Neuron-Implementation')
print('Working directory:', os.getcwd())

Working directory: /kaggle/working/H-Neuron-Implementation


In [4]:
# Colab/Kaggle setup. Restart the runtime if vLLM or torch dependencies require it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.9/360.9 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

In [5]:
# Fresh Qwen2.5-7B-Instruct rerun. Only the tokenizer is loaded here (via --tokenizer_path),
# so this cell itself is cheap; the memory concern only matters in h-neurons-1 where the LM runs.
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

# Set your OpenAI-compatible key only if you run extract_answer_tokens.py.
# In Colab: from google.colab import userdata; OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# In Kaggle: use Add-ons > Secrets, then read it with kaggle_secrets.
# OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_API_KEY')
# BASE_URL = os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1')

OUTPUT_DIR = 'data/small_subset_qwen7b'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('MODEL_ID =', MODEL_ID)

MODEL_ID = Qwen/Qwen2.5-1.5B-Instruct


## 2. Extract Qwen-tokenized answer tokens

This step uses an LLM API to select answer tokens, but tokenization is done with the Qwen tokenizer through `--tokenizer_path MODEL_ID`.

In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("api_key_openai")


In [7]:
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
BASE_URL_2 = "https://api.vietapi.tech/v1"
LLM_MODEL="models/gemini-3.1-flash-lite"
model = "gpt-5.4"

In [8]:
# WARNING: '500-truth/combined.jsonl' was produced from 1.5B responses. Answer-token indices must be extracted
# from the SAME model whose activations you will later probe, so this input needs to be regenerated on 7B
# before this cell is meaningful. The tokenizer path is now Qwen2.5-7B-Instruct so tokenization matches the 7B model.
!python h_neuron_scripts/extract_answer_tokens.py \
  --input_path "/kaggle/input/datasets/vkb0205/500-truth/combined.jsonl" \
  --output_path  "/kaggle/working/train_answer_tokens_small_qwen7b_500.jsonl" \
  --tokenizer_path "Qwen/Qwen2.5-7B-Instruct" \
  --api_key {api_key} \
  --base_url {BASE_URL_2} \
  --llm_model {model} \
  --resume

config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 1.71MB/s]
tokenizer_config.json: 7.30kB [00:00, 10.0MB/s]
vocab.json: 2.78MB [00:00, 54.6MB/s]
merges.txt: 1.67MB [00:00, 87.2MB/s]
tokenizer.json: 7.03MB [00:00, 93.3MB/s]
Using API key 1/1
Resume enabled: found 0 already processed IDs in /kaggle/working/train_answer_tokens_small_qwen_500.jsonl
Processing tokens: 0it [00:00, ?it/s]Saved tc_1643 to /kaggle/working/train_answer_tokens_small_qwen_500.jsonl
Processing tokens: 1it [00:06,  6.61s/it]Saved tc_1644 to /kaggle/working/train_answer_tokens_small_qwen_500.jsonl
Processing tokens: 2it [00:19, 10.07s/it]Saved tc_1648 to /kaggle/working/train_answer_tokens_small_qwen_500.jsonl
Processing tokens: 4it [00:22,  4.93s/it]Saved tc_1652 to /kaggle/working/train_answer_tokens_small_qwen_500.jsonl
Processing tokens: 6it [00:31,  4.72s/it]Saved tc_1655 to /kaggle/working/train_answer_tokens_small_qwen_500.jsonl
Processing tokens: 7it [00:41,  5.95s/it]Saved tc_1659 to /ka